# Добро пожаловать в Компьютерное зрение! #

Вы когда-нибудь хотели научить компьютер видеть? В этом курсе вы именно этим и займётесь!

В этом курсе вы:
- Используете современные глубокие нейронные сети для построения **классификатора изображений** с помощью Keras
- Спроектируете собственную **свёрточную сеть** из переиспользуемых блоков
- Изучите фундаментальные идеи визуального **извлечения признаков**
- Освоите искусство **трансферного обучения** для улучшения моделей
- Примените **аугментацию данных** для расширения набора данных

Если вы прошли курс *Введение в глубокое обучение*, у вас есть все необходимые знания для успешного прохождения.

Давайте начнём!

# Введение #

Этот курс познакомит вас с фундаментальными идеями компьютерного зрения. Наша цель — понять, как нейронная сеть может «понимать» естественное изображение достаточно хорошо, чтобы решать те же задачи, которые решает человеческая зрительная система.

Нейронные сети, которые лучше всего справляются с этой задачей, называются **свёрточными нейронными сетями** (иногда говорят **convnet** или **CNN**). Свёртка — это математическая операция, которая придаёт слоям свёрточной сети их уникальную структуру. В следующих уроках вы узнаете, почему эта структура так эффективна для решения задач компьютерного зрения.

Мы применим эти идеи к задаче **классификации изображений**: сможем ли мы обучить компьютер определять, что изображено на картинке? Возможно, вы видели [приложения](https://identify.plantnet.org/), которые могут определить вид растения по фотографии. Это и есть классификатор изображений! В этом курсе вы научитесь создавать классификаторы изображений, не уступающие тем, что используются в профессиональных приложениях.

Хотя основное внимание мы уделим классификации изображений, то, что вы узнаете в этом курсе, применимо к любым задачам компьютерного зрения. В конце вы будете готовы перейти к более продвинутым приложениям, таким как генеративно-состязательные сети и сегментация изображений.

# Свёрточный классификатор #

Свёрточная сеть, используемая для классификации изображений, состоит из двух частей: **свёрточной основы** и **полносвязной головы**.

<center>
<!-- <img src="./images/1-parts-of-a-convnet.png" width="600" alt="The parts of a convnet: image, base, head, class; input, extract, classify, output.">-->
<img src="./img/U0n5xjU.png" width="600" style="background-color: white;" alt="Части свёрточной сети: изображение, основа, голова, класс; вход, извлечение, классификация, выход.">
</center>

Основа используется для **извлечения признаков** из изображения. Она состоит в основном из слоёв, выполняющих операцию свёртки, но часто включает и другие типы слоёв. (Вы узнаете о них в следующем уроке.)

Голова используется для **определения класса** изображения. Она состоит в основном из полносвязных слоёв, но может включать и другие слои, например dropout.

Что мы подразумеваем под визуальным признаком? Признаком может быть линия, цвет, текстура, форма, паттерн — или какое-то сложное сочетание.

Весь процесс выглядит примерно так:

<center>
<!-- <img src="./images/1-extract-classify.png" width="600" alt="The idea of feature extraction."> -->
<img src="./img/UUAafkn.png" width="600" alt="Идея извлечения признаков.">
</center>

Реально извлекаемые признаки выглядят несколько иначе, но это передаёт общую идею.

# Обучение классификатора #

Цель сети во время обучения — выучить две вещи:
1. какие признаки извлекать из изображения (основа),
2. какой класс соответствует каким признакам (голова).

В наши дни свёрточные сети редко обучают с нуля. Чаще мы **переиспользуем основу предобученной модели**. К предобученной основе мы затем **присоединяем необученную голову**. Другими словами, мы переиспользуем часть сети, которая уже научилась *1. Извлекать признаки*, и присоединяем к ней новые слои для изучения *2. Классификации*.

<center>
<!-- <img src="./images/1-attach-head-to-base.png" width="400" alt="Attaching a new head to a trained base."> -->
<img src="./img/E49fsmV.png" width="400" alt="Присоединение новой головы к обученной основе.">
</center>

Поскольку голова обычно состоит всего из нескольких полносвязных слоёв, можно создавать очень точные классификаторы на относительно небольших объёмах данных.

Переиспользование предобученной модели — это техника, известная как **трансферное обучение**. Она настолько эффективна, что сегодня почти каждый классификатор изображений её использует.

# Пример — Обучение свёрточного классификатора #

На протяжении этого курса мы будем создавать классификаторы, пытающиеся решить следующую задачу: это изображение *Автомобиля* или *Грузовика*? Наш набор данных содержит около 10 000 изображений различных автомобилей, примерно поровну легковых и грузовых.

## Шаг 1 — Загрузка данных ##

Следующая скрытая ячейка импортирует библиотеки и настроит конвейер данных. У нас есть обучающая выборка `ds_train` и проверочная выборка `ds_valid`.

In [2]:

# Imports
import os, warnings
import matplotlib.pyplot as plt
from matplotlib import gridspec

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory


# Reproducability
def set_seed(seed=31415):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
set_seed(31415)

# Set Matplotlib defaults
plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large',
       titleweight='bold', titlesize=18, titlepad=10)
plt.rc('image', cmap='magma')
warnings.filterwarnings("ignore") # to clean up output cells


# Load training and validation sets
ds_train_ = image_dataset_from_directory(
    './data/train',
    labels='inferred',
    label_mode='binary',
    image_size=[128, 128],
    interpolation='nearest',
    batch_size=64,
    shuffle=True,
)
ds_valid_ = image_dataset_from_directory(
    './data/valid',
    labels='inferred',
    label_mode='binary',
    image_size=[128, 128],
    interpolation='nearest',
    batch_size=64,
    shuffle=False,
)

# Data Pipeline
def convert_to_float(image, label):
    image = tf.image.convert_image_dtype(image, dtype=tf.float32)
    return image, label

AUTOTUNE = tf.data.experimental.AUTOTUNE
ds_train = (
    ds_train_
    .map(convert_to_float)
    .cache()
    .prefetch(buffer_size=AUTOTUNE)
)
ds_valid = (
    ds_valid_
    .map(convert_to_float)
    .cache()
    .prefetch(buffer_size=AUTOTUNE)
)


Found 5117 files belonging to 2 classes.
Found 5051 files belonging to 2 classes.


Давайте взглянем на несколько примеров из обучающего набора.

In [3]:

import matplotlib.pyplot as plt

## Шаг 2 — Определение предобученной основы ##

Наиболее часто используемый набор данных для предобучения — [*ImageNet*](http://image-net.org/about-overview), большой набор изображений множества различных природных объектов. Keras включает множество моделей, предобученных на ImageNet, в своём [`applications` модуле](https://www.tensorflow.org/api_docs/python/tf/keras/applications). Предобученная модель, которую мы будем использовать, называется **VGG16**.

In [10]:
# pretrained_base = tf.keras.models.load_model(
#     './data/cv-course-models/vgg16-pretrained-base',
# )
# pretrained_base.trainable = False
from tensorflow.keras.applications import VGG16

# Загружаем предобученную VGG16 из официальных весов
pretrained_base = VGG16(
    weights='imagenet',  # или None для необученной
    include_top=False,   # убираем полносвязные слои
    input_shape=(128, 128, 3)
)

pretrained_base.trainable = False

## Шаг 3 — Присоединение головы ##

Далее мы присоединяем голову классификатора. В этом примере мы используем слой скрытых нейронов (первый слой `Dense`), за которым следует слой для преобразования выходов в оценку вероятности для класса 1 — `Truck`. Слой `Flatten` преобразует двумерные выходы основы в одномерные входные данные, необходимые голове.

In [11]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    pretrained_base,
    layers.Flatten(),
    layers.Dense(6, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])

## Шаг 4 — Обучение ##

Наконец, давайте обучим модель. Поскольку это задача бинарной классификации, мы будем использовать бинарные версии `crossentropy` и `accuracy`. Оптимизатор `adam` обычно показывает хорошие результаты, поэтому мы выберем его.

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy'],
)

history = model.fit(
    ds_train,
    validation_data=ds_valid,
    epochs=30,
    verbose=0,
)

При обучении нейронной сети всегда полезно просматривать графики функции потерь и метрик. Объект `history` содержит эту информацию в словаре `history.history`. Мы можем использовать Pandas для преобразования этого словаря в DataFrame и построить график встроенным методом.

In [ ]:
import pandas as pd

history_frame = pd.DataFrame(history.history)
history_frame.loc[:, ['loss', 'val_loss']].plot()
history_frame.loc[:, ['binary_accuracy', 'val_binary_accuracy']].plot();

# Заключение #

На этом уроке мы узнали о структуре свёрточного классификатора: **голова** действует как классификатор поверх **основы**, которая выполняет извлечение признаков.

Голова, по сути, представляет собой обычный классификатор, подобный тому, с которым вы познакомились во вводном курсе. В качестве признаков она использует те, что были извлечены основой. В этом и заключается основная идея свёрточных классификаторов: мы можем присоединить блок, выполняющий инженерию признаков, непосредственно к самому классификатору.

Это одно из главных преимуществ глубоких нейронных сетей перед традиционными моделями машинного обучения: при правильной структуре сети глубокая нейронная сеть может самостоятельно научиться конструировать признаки, необходимые для решения своей задачи.

В следующих нескольких уроках мы рассмотрим, как свёрточная основа осуществляет извлечение признаков. Затем вы научитесь применять эти идеи и создавать собственные классификаторы.

# Ваша очередь #

А сейчас переходите к [**Упражнению**](https://www.kaggle.com/kernels/fork/10781907) и создайте собственный классификатор изображений!

---




*Есть вопросы или комментарии? Посетите [форум обсуждения курса](https://www.kaggle.com/learn/computer-vision/discussion), чтобы пообщаться с другими учащимися.*